# F4 primer: just enough PyTorch to read Karpathy's GPT code

Throwaway learning code. Run the cells top to bottom (kernel: project `.venv`).

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(42)  # same "random" numbers every run, so output is reproducible


def show(name, t):
    print(f"{name}  shape={tuple(t.shape)}\n{t}\n")

## 1. Tensors and `(B, T)` shapes
Tensors are numpy arrays that can live on a GPU and track gradients.

In [ ]:
B, T = 2, 4  # 2 snippets in a batch, each 4 tokens long
idx = torch.tensor([[5, 2, 7, 1],  # snippet 0: token ids
                    [3, 3, 0, 6]])  # snippet 1
show("idx (token ids)", idx)  # shape (B, T): this is what the data loader yields

## 2. `nn.Embedding`: id → vector (adds the C dimension)
A learnable lookup table.

In [ ]:
vocab_size, C = 8, 3  # 8 possible tokens, each becomes a 3-number vector
emb = nn.Embedding(vocab_size, C)  # a (vocab_size, C) table of random numbers...
show("embedding table (learned during training)", emb.weight.data)

In [ ]:
x = emb(idx)  # ...and calling it just looks up rows
show("x = emb(idx)   <- (B, T) became (B, T, C)", x.data)
print("token 3 appears twice in snippet 1 (same id -> same vector):")
print(x[1, 0].data, x[1, 1].data)

## 3. `nn.Linear`: `y = x @ W.T + b`, applied to the last dim

In [ ]:
head_size = 2
proj = nn.Linear(C, head_size, bias=False)  # W is (head_size, C) = learned
y = proj(x)
show("y = proj(x)   <- (B, T, C) became (B, T, head_size)", y.data)

This is exactly what Karpathy's key/query/value layers are: three `nn.Linear(C, head_size)` → three different learned views of each token.

## 4. The trick: each token = average of itself + earlier tokens
Three ways to compute the same thing.

In [ ]:
xb = x[0].data  # take snippet 0 only, shape (T, C)

# Version 1: obvious but slow python loop
v1 = torch.stack([xb[: t + 1].mean(dim=0) for t in range(T)])
show("v1 (loop)", v1)

In [ ]:
# Version 2: lower-triangular matrix of weights, one matrix multiply
tril = torch.tril(torch.ones(T, T))
show("tril (1 = 'may look at')", tril)
wei = tril / tril.sum(dim=1, keepdim=True)
show("wei = each row sums to 1 -> 'how much to take from each token'", wei)
v2 = wei @ xb  # (T, T) @ (T, C) -> (T, C)
show("v2 (matmul)", v2)

In [ ]:
# Version 3: same thing written with softmax -- this is the form attention uses
scores = torch.zeros(T, T)  # attention will put REAL q.k scores here instead of 0
scores = scores.masked_fill(tril == 0, float("-inf"))  # future = -inf
show("scores after mask (-inf = cannot see the future)", scores)
wei3 = F.softmax(scores, dim=-1)  # exp(-inf)=0, so the future gets weight 0
v3 = wei3 @ xb
show("v3 (softmax)", v3)

In [ ]:
print("all three versions equal?", torch.allclose(v1, v2), torch.allclose(v1, v3))

**The key idea:** in v3 the scores are all zeros → plain average. Self-attention replaces those zeros with q·k scores, so each token chooses *which* past tokens to listen to. That swap is your `Head.forward()` in F4 step 3.